# Loading ERA5 Data and Merging with eaglei

> *Pre-requisites*: Code requires most of the packages listed [here](https://github.com/google-research/arco-era5/tree/main/docs/environment.yml):

We will download ERA5 variables from the Copernicus Climate Data Store

In [4]:
import cdsapi

dataset = "reanalysis-era5-land"
request = {
    "variable": [
        "2m_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "snowfall",
        "total_precipitation"
    ],
    "year": ["2014","2015","2016","2017","2018","2019","2020","2021","20122","2023"],
    "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "data_format": "grib",
    "download_format": "zip",
    "area": [50, -125, 24, -66]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2025-03-29 05:11:28,682 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-03-29 05:11:28,683 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-land/execution
cost limits exceeded
Your request is too large, please reduce your selection.

In [17]:
#Load the ERA5_2018.grib file from Data as an xarray
import xarray as xr
ERA5_2018 = xr.open_dataset('../Data/ERA5_2018.grib', decode_times=False)

Ignoring index file '../Data/ERA5_2018.grib.5b7b6.idx' older than GRIB file


In [18]:
#Remove the coordinates number, surface, and valid_time from ERA5_2018
ERA5_2018 = ERA5_2018.drop(['number', 'surface', 'valid_time'])

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_40403/2423029272.py:2: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ERA5_2018 = ERA5_2018.drop(['number', 'surface', 'valid_time'])


In [21]:
ERA5_2018

<xarray.Dataset> Size: 5GB
Dimensions:    (time: 366, step: 4, latitude: 261, longitude: 591)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2018-01-01 ... 2018-04-02T06:00:00
  * step       (step) float64 32B 6.0 12.0 18.0 24.0
  * latitude   (latitude) float64 2kB 50.0 49.9 49.8 49.7 ... 24.2 24.1 24.0
  * longitude  (longitude) float64 5kB -125.0 -124.9 -124.8 ... -66.1 -66.0
Data variables:
    t2m        (time, step, latitude, longitude) float32 903MB ...
    u10        (time, step, latitude, longitude) float32 903MB ...
    v10        (time, step, latitude, longitude) float32 903MB ...
    sf         (time, step, latitude, longitude) float32 903MB ...
    tp         (time, step, latitude, longitude) float32 903MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-19T08:22 GRIB to CDM+CF via cfgrib-0.9.1...

In [20]:
#In ERA5_2018 convert the time coordinate from int64 to datetime values starting at January 1st, 2018
ERA5_2018['time'] = pd.date_range(start='2018-01-01', periods=len(ERA5_2018['time']), freq='6H')

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_40403/2016527515.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ERA5_2018['time'] = pd.date_range(start='2018-01-01', periods=len(ERA5_2018['time']), freq='6H')


In [3]:
#Use pandas to load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet data set
import pandas as pd
eaglei_outages = pd.read_parquet('../Data/eaglei_data/eaglei_outages_with_county_info.parquet')

In [4]:
#Restrict eaglei_outages to YEAR 2018
eaglei_outages = eaglei_outages[eaglei_outages['YEAR'] == 2018]

In [5]:
eaglei_outages.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4108373 entries, 4222117 to 8330489
Data columns (total 19 columns):
 #   Column                            Dtype         
---  ------                            -----         
 0   fips_code                         float64       
 1   customers_out                     float64       
 2   datetime                          datetime64[ns]
 3   YEAR                              int32         
 4   NAME                              object        
 5   STUSPS                            object        
 6   FIPS                              int64         
 7   Pct_Buried_Lines                  float64       
 8   neighbors                         object        
 9   Subregion                         object        
 10  centroid_longitude                float64       
 11  centroid_latitude                 float64       
 12  centroid_rounded                  object        
 13  POPULATION                        float64       
 14  BUILDVALUE       

In [22]:
import numpy as np

def get_ds_mean(t,h, x,y,ds):
    """
    Parameters:
    -t,x,y: start time, latitude, longitude
    -ds: xarray dataset
    Returns:
    A list containing the features of ds at specified start time and position
    """
    features = list(ds.keys())
    if not any(np.isnan([x, y])):   
        temp = ds.sel(time=t, step = h, latitude=x, longitude=y, method='nearest')
        return [temp[key].values for key in features]
    else:
        return [np.nan for key in features]


features = list(eaglei_outages.keys())
eaglei_outages[features] = pd.DataFrame(eaglei_outages.apply(lambda x:get_ds_mean(x['datetime'],6, x['centroid_latitude'],x['centroid_longitude'],ERA5_2018),axis=1).tolist(),index=eaglei_outages.index) #takes about 24 minutes to run through a year of df


KeyboardInterrupt: 

In [8]:
eaglei_outages.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4108373 entries, 4222117 to 8330489
Data columns (total 19 columns):
 #   Column                            Dtype         
---  ------                            -----         
 0   fips_code                         float64       
 1   customers_out                     float64       
 2   datetime                          datetime64[ns]
 3   YEAR                              int32         
 4   NAME                              object        
 5   STUSPS                            object        
 6   FIPS                              int64         
 7   Pct_Buried_Lines                  float64       
 8   neighbors                         object        
 9   Subregion                         object        
 10  centroid_longitude                float64       
 11  centroid_latitude                 float64       
 12  centroid_rounded                  object        
 13  POPULATION                        float64       
 14  BUILDVALUE       

In [10]:
#Convert eaglei_outages to an xarray DataArray with longitude, latitude, and datetime as coordinates and using all its variables as data variables
eaglei_outages = xr.DataArray(eaglei_outages, coords=[('longitude', eaglei_outages['centroid_longitude']), ('latitude', eaglei_outages['centroid_latitude']), ('time', eaglei_outages['datetime'])])

ValueError: coords is not dict-like, but it has 3 items, which does not match the 2 dimensions of the data